## Spark warmup

🎯 The goal of the exercise is to practice some pyspark syntax and will involve 5 questions tested directly in this jupyter notebook!

## Setup

The next cell will create the spark session and load the data in for you

In [39]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Olist Exercises").getOrCreate()

df_customers = spark.read.csv("data/olist_customers_dataset.csv", header=True, inferSchema=True)
df_geolocation = spark.read.csv("data/olist_geolocation_dataset.csv", header=True, inferSchema=True)
df_items = spark.read.csv("data/olist_order_items_dataset.csv", header=True, inferSchema=True)
df_payments = spark.read.csv("data/olist_order_payments_dataset.csv", header=True, inferSchema=True)
df_reviews = spark.read.csv("data/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("data/olist_orders_dataset.csv", header=True, inferSchema=True)
df_products = spark.read.csv("data/olist_products_dataset.csv", header=True, inferSchema=True)
df_sellers = spark.read.csv("data/olist_sellers_dataset.csv", header=True, inferSchema=True)
df_category_translation = spark.read.csv("data/product_category_name_translation.csv", header=True, inferSchema=True)


**Question 1:** Caculate the count of each of the order statuses and save the result as `df_order_status_count`


In [40]:
# Calcular la cantidad de cada estado de los pedidos
df_order_status_count = df_orders.groupBy("order_status").count()

# Mostrar el resultado
df_order_status_count.show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  625|
|    invoiced|  314|
|     created|    5|
|   delivered|96478|
| unavailable|  609|
|  processing|  301|
|    approved|    2|
+------------+-----+



🧪 **Testing** Run the cell below

In [41]:
from nbresult import ChallengeResult

pandas_df_order_status_count = df_order_status_count.toPandas()

result = ChallengeResult('order_status_count',
    order_status_count_df=pandas_df_order_status_count
)
result.write()
print(result.check())


**Question 2:** Calculate the Customer Lifetime Value for each customer. CLV is calculated as the sum of the payment values for each customer. Save the resulting spark DataFrame into a variable called `df_clv`. The clv column should be named `CLV`.


In [42]:
# Realiza el join entre df_payments y df_orders usando 'order_id'
df_combined = df_payments.join(df_orders, on="order_id", how="inner")

# Calcular el CLV sumando los valores de pago por customer_id
df_clv = df_combined.groupBy("customer_id").sum("payment_value").withColumnRenamed("sum(payment_value)", "CLV")

# Mostrar el resultado
df_clv.show()

+--------------------+------------------+
|         customer_id|               CLV|
+--------------------+------------------+
|f54a9f0e6b351c431...|             35.95|
|2a1dfb647f32f4390...|            508.17|
|4f28355e5c17a4a42...|            153.72|
|4632eb5a8f175f6fe...|             91.66|
|843ff05b30ce4f75b...|            102.74|
|a4156bb8aff5d6722...|            206.87|
|1099d033c74a027a7...|              61.7|
|7a3bd3b37285f0ab2...|            473.12|
|5fff39f1b59dc4d2f...|             90.58|
|18f6ca10777417c93...|             61.36|
|f1e46939e6408b3e6...|              53.0|
|a8695124db570d100...|             85.24|
|545b9a267af9ba134...|            130.83|
|90d7075599361b694...|             84.28|
|a340ce6c3570e68d4...|            122.69|
|ce0681e1c3f70e145...|             82.73|
|ee905ec97794ec6e9...|170.23000000000002|
|b74ca180d63f9ae04...|             98.62|
|0de46efc7d10114ac...|             124.8|
|8bdae6b4ff9bc7f4c...|            376.56|
+--------------------+------------

🧪 **Testing** Run the cell below

In [43]:
from nbresult import ChallengeResult

df_clv_pd = df_clv.toPandas()

result = ChallengeResult('customer_clv',
    df_clv_pd=df_clv_pd
)
result.write()
print(result.check())


**Question 3:** Calculate the total revenue for each seller. Save the resulting spark DataFrame in a variable called `df_seller_revenue`. The total revenue column should be named `Total_Revenue`.

In [44]:
# Realiza el join entre df_orders y df_payments usando 'order_id'
df_combined = df_orders.join(df_payments, on="order_id", how="inner")

# Calcular el ingreso total por vendedor usando 'seller_id' desde df_orders
df_seller_revenue = df_combined.join(df_items, on="order_id", how="inner") \
    .groupBy("seller_id").sum("payment_value") \
    .withColumnRenamed("sum(payment_value)", "Total_Revenue")

# Mostrar el resultado
df_seller_revenue.show()

+--------------------+------------------+
|           seller_id|     Total_Revenue|
+--------------------+------------------+
|062ce95fa2ad4dfae...|          10730.16|
|0ea22c1cfbdc755f8...|20780.680000000004|
|a3082f442524a1be4...| 656.6700000000001|
|8e6cc767478edae94...|          10154.07|
|e63e8bfa530fb1691...|           2311.44|
|ff063b022a9a0aab9...|          14862.57|
|9803a40e82e45418a...|           21265.2|
|ec8879960bd2221d5...|           4881.38|
|a49928bcdf77c55c6...|12521.220000000001|
|da7039f29f90ce5b4...|1873.7599999999998|
|6eeed17989b0ae47c...|            819.53|
|9c068d10aca38e85c...|3771.6000000000004|
|4d600e08ecbe08258...|           4633.46|
|791cfcfe22fe4a771...|            653.75|
|2009a095de2a2a416...|           1275.73|
|9b1050e85becf3ae9...|            115.34|
|c522be04e020c1e7b...|             838.3|
|b3f19518fcec265b2...|            624.48|
|33ac3e28642ab8bda...| 4491.509999999999|
|71039d19d4303bf90...|           4462.25|
+--------------------+------------

In [45]:
from nbresult import ChallengeResult

df_seller_revenue_pd = df_seller_revenue.toPandas()

result = ChallengeResult('seller_revenue',
    df_seller_revenue_pd=df_seller_revenue_pd
)
result.write()
print(result.check())


**Question 4:** Identify the top 10 product categories with the highest **count** of canceled or unavailable orders. Save the result in a spark DataFrame called `df_high_cancel_categories`. The column for total instances should be named `Total_Canceled_Unavailable`


In [46]:
# Filtrar los pedidos cancelados
df_canceled_orders = df_orders.filter(df_orders.order_status == "canceled")

# Contar el número de pedidos cancelados por categoría
df_high_cancel_categories = df_canceled_orders.join(df_items, on="order_id", how="inner") \
    .join(df_products, on="product_id", how="inner") \
    .groupBy("product_category_name").count() \
    .withColumnRenamed("count", "Total_Canceled_Unavailable") \
    .orderBy("Total_Canceled_Unavailable", ascending=False) \
    .limit(10)

# Mostrar el resultado
df_high_cancel_categories.show()

+---------------------+--------------------------+
|product_category_name|Total_Canceled_Unavailable|
+---------------------+--------------------------+
|        esporte_lazer|                        51|
| utilidades_domest...|                        49|
| informatica_acess...|                        46|
|     moveis_decoracao|                        36|
|         beleza_saude|                        36|
|           brinquedos|                        34|
|           automotivo|                        30|
|   relogios_presentes|                        21|
|                bebes|                        20|
|   ferramentas_jardim|                        19|
+---------------------+--------------------------+



In [47]:
from nbresult import ChallengeResult

df_high_cancel_categories_pd = df_high_cancel_categories.toPandas()

result = ChallengeResult('high_cancel_categories',
    df_high_cancel_categories_pd=df_high_cancel_categories_pd
)
result.write()
print(result.check())


**Question 5:** Calculate the average extended delivery time for each product category. The extended delivery time is the actual delivery time minus the estimated delivery time. Save the result in a variable called `df_delivery_time`. The `Avg_Extended_Delivery` column should contain the number of hours (as a float), sorted in descending order, and limited to the top 10 product categories with the highest average extended delivery time.

<details>
    <summary>💡Hint</summary>

Break up the transformation into a few steps! Try creating an `actual_delivery_time` and `estimated_delivery_time` on a DataFrame that makes sense.

</details>

In [48]:
from pyspark.sql.functions import col, unix_timestamp

# Calcular el tiempo de entrega real y estimado
df_delivery_time = df_orders.join(df_items, on="order_id", how="inner") \
    .join(df_products, on="product_id", how="inner") \
    .withColumn("order_purchase_timestamp", unix_timestamp("order_purchase_timestamp")) \
    .withColumn("order_delivered_customer_date", unix_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", unix_timestamp("order_estimated_delivery_date")) \
    .withColumn("actual_delivery_time", (col("order_delivered_customer_date") - col("order_purchase_timestamp")) / 3600) \
    .withColumn("estimated_delivery_time", (col("order_estimated_delivery_date") - col("order_purchase_timestamp")) / 3600) \
    .withColumn("extended_delivery_time", col("actual_delivery_time") - col("estimated_delivery_time"))

# Calcular el tiempo promedio de entrega extendido por categoría de producto
df_delivery_time = df_delivery_time.groupBy("product_category_name") \
    .avg("extended_delivery_time") \
    .withColumnRenamed("avg(extended_delivery_time)", "Avg_Extended_Delivery") \
    .orderBy("Avg_Extended_Delivery", ascending=False) \
    .limit(10)

# Mostrar el resultado
df_delivery_time.show()

+---------------------+---------------------+
|product_category_name|Avg_Extended_Delivery|
+---------------------+---------------------+
|   artes_e_artesanato|   -146.2061226851852|
| moveis_colchao_e_...|  -155.48177927927927|
|      casa_conforto_2|  -186.99555555555554|
| portateis_cozinha...|  -210.37928571428571|
|             pc_gamer|  -215.15850694444447|
|        casa_conforto|  -218.75630924630917|
|            alimentos|  -220.29769483411272|
|                audio|  -227.19191068139958|
| fashion_underwear...|  -245.05284339457566|
|    livros_importados|  -250.30922514619883|
+---------------------+---------------------+



In [49]:
from nbresult import ChallengeResult

df_delivery_time_pd = df_delivery_time.toPandas()

result = ChallengeResult('delivery_time',
    df_delivery_time_pd=df_delivery_time_pd
)
result.write()
print(result.check())


## Finished 🏁

Now you have finished the notebook you should have a good grounding of how to use pyspark!

You can run 

```bash
make test
```

Make sure to commit and push all your changes to github so you can track your progress on Kitt!

Time to move on to the next exercise!